In [1]:
import re
import time
import random
import html
import requests
import pandas as pd
import os
from getpass import getpass
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import quote
from selenium.common.exceptions import TimeoutException

### 1. Get all the papers from these journals

In [2]:
#RpSFzYyBpdb0Rrq8rB7hyw
API_KEY = getpass("Enter your OpenAlex API key: ")

url = "https://api.openalex.org/sources"

params = {
    "filter": "issn:0732-2399|1526-548X|0025-1909|1526-55010022-2437|1547-7193|0022-2429|1547-7185|0093-5301|1537-5277|1570-7156|1573-711X",
    "per_page": 20,
    "api_key": API_KEY
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

for journal in data["results"]:
    print("Journal:", journal["display_name"])
    print("OpenAlex ID:", journal["id"])
    print("ISSNs:", journal["issn"])
    print("-" * 50)

Enter your OpenAlex API key:  ········


Journal: Management Science
OpenAlex ID: https://openalex.org/S33323087
ISSNs: ['0025-1909', '1526-5501']
--------------------------------------------------
Journal: Journal of Marketing
OpenAlex ID: https://openalex.org/S142990027
ISSNs: ['0022-2429', '1547-7185']
--------------------------------------------------
Journal: Journal of Marketing Research
OpenAlex ID: https://openalex.org/S119950638
ISSNs: ['0022-2437', '1547-7193']
--------------------------------------------------
Journal: Journal of Consumer Research
OpenAlex ID: https://openalex.org/S145429826
ISSNs: ['0093-5301', '1537-5277']
--------------------------------------------------
Journal: Marketing Science
OpenAlex ID: https://openalex.org/S163534328
ISSNs: ['0732-2399', '1526-548X']
--------------------------------------------------
Journal: Quantitative Marketing and Economics
OpenAlex ID: https://openalex.org/S4210206015
ISSNs: ['1570-7156', '1573-711X']
--------------------------------------------------


In [3]:
journals = {
    "MS": {
        "name": "Marketing Science",
        "id": "S163534328"
    },
    "MGSC": {
        "name": "Management Science",
        "id": "S33323087"
    },
    "JMR": {
        "name": "Journal of Marketing Research",
        "id": "S119950638"
    },
    "JM": {
        "name": "Journal of Marketing",
        "id": "S142990027"
    },
    "JCR": {
        "name": "Journal of Consumer Research",
        "id": "S145429826"
    },
    "QME": {
        "name": "Quantitative Marketing and Economics",
        "id": "S4210206015"
    }
}

In [4]:
def get_journal_papers(journal_code, journal_name, journal_id):

    url = "https://api.openalex.org/works"

    params = {
        "filter": (
            f"primary_location.source.id:{journal_id},"
            "publication_year:2022-2026,"
            "type:article"
        ),
        "per_page": 100,
        "cursor": "*",
        "select": (
            "id,doi,title,publication_year,"
            "abstract_inverted_index,authorships"
        ),
        "api_key": API_KEY
    }

    all_papers = []

    while True:

        response = requests.get(url, params=params)
        response.raise_for_status()

        data = response.json()

        for paper in data["results"]:

            # Reconstruct abstract
            inverted_index = paper.get("abstract_inverted_index")
            abstract = None

            if inverted_index:

                words = {}

                for word, positions in inverted_index.items():
                    for position in positions:
                        words[position] = word

                abstract = " ".join(
                    words[i] for i in sorted(words)
                )

            # Extract authors
            authorships = paper.get("authorships", [])

            authors = [
                authorship["author"]["display_name"]
                for authorship in authorships
                if authorship.get("author")
            ]

            all_papers.append({
                "openalex_id": paper.get("id"),
                "doi": paper.get("doi"),
                "journal_code": journal_code,
                "journal": journal_name,
                "title": paper.get("title"),
                "year": paper.get("publication_year"),
                "authors": authors,
                "abstract_openalex": abstract
            })

        # Check for next page
        next_cursor = data["meta"]["next_cursor"]

        if not next_cursor:
            break

        params["cursor"] = next_cursor

    return all_papers

In [5]:
all_papers = []

for code, journal in journals.items():

    print(f"Collecting: {journal['name']}...")

    papers = get_journal_papers(
        journal_code=code,
        journal_name=journal["name"],
        journal_id=journal["id"]
    )

    all_papers.extend(papers)

    print(f"Retrieved {len(papers)} papers")
    print("-" * 50)

Collecting: Marketing Science...
Retrieved 322 papers
--------------------------------------------------
Collecting: Management Science...
Retrieved 1768 papers
--------------------------------------------------
Collecting: Journal of Marketing Research...
Retrieved 252 papers
--------------------------------------------------
Collecting: Journal of Marketing...
Retrieved 260 papers
--------------------------------------------------
Collecting: Journal of Consumer Research...
Retrieved 299 papers
--------------------------------------------------
Collecting: Quantitative Marketing and Economics...
Retrieved 50 papers
--------------------------------------------------


In [6]:
df = pd.DataFrame(all_papers)

df = df.sort_values(
    by=["journal_code", "year", "title"],
    ascending=[True, False, True]
).reset_index(drop=True)
df.head()

,openalex_id,doi,journal_code,journal,title,year,authors,abstract_openalex
0,https://openalex.org/W7154685842,https://doi.org/10.1093/jcr/ucag010,JCR,Journal of Consumer Research,A Framework for Understanding Consumer Respons...,2026,"[Cory Haltman, Jianna Jin, Grant E. Donnelly, ...","Abstract In recent years, firms’ depictions of..."
1,https://openalex.org/W7203715795,https://doi.org/10.1093/jcr/ucag027,JCR,Journal of Consumer Research,A Mega-Replication of the Effect of Cash Versu...,2026,"[Christopher J. Bechler, Rhia Catapano, Szu‐ch...",Abstract This study revisits the widely accept...
2,https://openalex.org/W7141561795,https://doi.org/10.1093/jcr/ucag001,JCR,Journal of Consumer Research,A Word of Thanks,2026,[],NaN
3,https://openalex.org/W7172001493,https://doi.org/10.1093/jcr/ucag025,JCR,Journal of Consumer Research,Are Critics More Credible? When Positivity Is ...,2026,"[Junha Kim, Joseph K. Goodman]","Abstract When writing a review, influencers ma..."
4,https://openalex.org/W7211969452,https://doi.org/10.1093/jcr/ucag035,JCR,Journal of Consumer Research,Consumer Entrapment,2026,"[Fredah Mwiti, Maria Piacentini, Andrew D. Pre...",Abstract Why do consumers sometimes remain tra...


In [7]:
# Total papers retrieved
print("Total papers retrieved:", len(df))

# Papers with abstracts
print("Papers with abstracts:", df["abstract_openalex"].notna().sum())

# Papers missing abstracts
print("Missing abstracts:", df["abstract_openalex"].isna().sum())

# Publication year distribution
print("\nPapers by year:")
print(df["year"].value_counts().sort_index())

# Check duplicate titles
print("\nDuplicate titles:", df["title"].duplicated().sum())

Total papers retrieved: 2951
Papers with abstracts: 2772
Missing abstracts: 179

Papers by year:
year
2022    611
2023    534
2024    588
2025    690
2026    528
Name: count, dtype: int64

Duplicate titles: 34


### 2. Filter based on keywords

In [8]:
KEYWORDS = ["artificial intelligence","AI","machine learning","ML","deep learning",
            "large language model","large language models","LLM","LLMs"
]

In [9]:
def find_keywords(abstract):

    if pd.isna(abstract):
        return []

    found_keywords = []

    for keyword in KEYWORDS:

        pattern = r"\b" + re.escape(keyword) + r"\b"

        if re.search(pattern, abstract, flags=re.IGNORECASE):
            found_keywords.append(keyword)

    return found_keywords

In [10]:
df["matched_keywords"] = df["abstract_openalex"].apply(find_keywords)
df_filtered = df[
    df["matched_keywords"].apply(len) > 0
].copy()
df_filtered = df_filtered[
    ["title", "journal_code", "year", "authors", "matched_keywords", "abstract_openalex", "doi"]
]
df_filtered = df_filtered.reset_index(drop=True)
len(df_filtered)

205

### 3. Retrieve the full abstract 
Note that OpenAlex doesn't have full abstract so we need to go into Google Scholar and retrieve the abstract there

In [14]:
def get_scholar_data(driver, doi):

    result = {
        "google_scholar_title": None,
        "google_scholar_abstract": None,
        "google_scholar_citations": None,
        "google_scholar_status": None
    }

    if pd.isna(doi):
        result["google_scholar_status"] = "Missing DOI"
        return result

    # Remove DOI prefix
    doi_clean = doi.replace("https://doi.org/", "").strip()

    # Build Google Scholar search URL
    search_url = (
        "https://scholar.google.com/scholar?q="
        + quote(doi_clean)
    )

    try:
        driver.get(search_url)

        # Wait for search results
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "div.gs_r.gs_or.gs_scl")
            )
        )

        results = driver.find_elements(
            By.CSS_SELECTOR, "div.gs_r.gs_or.gs_scl"
        )

        if not results:
            result["google_scholar_status"] = "No results"
            return result

        # Use the first search result
        paper = results[0]

        # Extract title
        try:
            result["google_scholar_title"] = paper.find_element(
                By.CSS_SELECTOR, "h3.gs_rt"
            ).text
        except:
            pass

        # Extract abstract
        try:
            abstract_element = paper.find_element(
                By.CSS_SELECTOR,
                "div.gs_fma_abs div.gs_fma_snp div.gsh_csp"
            )

            result["google_scholar_abstract"] = (
                abstract_element.text.strip()
            )

        except:
            result["google_scholar_abstract"] = None

        # Extract citation count
        try:
            citation_element = paper.find_element(
                By.CSS_SELECTOR, "a.gs_or_cited"
            )

            citation_text = citation_element.text

            result["google_scholar_citations"] = int(
                citation_text.replace("Cited by", "").strip()
            )

        except:
            result["google_scholar_citations"] = None

        # Determine retrieval status
        if result["google_scholar_abstract"]:
            result["google_scholar_status"] = "Success"
        else:
            result["google_scholar_status"] = "Abstract not found"

    except TimeoutException:
        result["google_scholar_status"] = "Timeout or blocked"

    except Exception as e:
        result["google_scholar_status"] = str(e)

    return result

In [17]:
# Initial round scraping
driver = webdriver.Chrome()

scholar_results = []

consecutive_failures = 0
max_consecutive_failures = 3

for i, row in df_filtered.iterrows():

    print(f"Processing paper {i + 1}/{len(df_filtered)}...")

    result = get_scholar_data(driver, row["doi"])
    result["doi"] = row["doi"]

    scholar_results.append(result)

    # Check scraping status
    if result["google_scholar_status"] == "Success":
        consecutive_failures = 0
    else:
        consecutive_failures += 1
        print(f"Consecutive failures: {consecutive_failures}")

    # Stop after more than 3 consecutive failures
    if consecutive_failures > max_consecutive_failures:
        print("More than 3 consecutive failures. Stopping scraping.")
        break

    time.sleep(random.uniform(5, 15))

# Close Chrome
driver.quit()

# Save all retrieved results
df_round1 = pd.DataFrame(scholar_results)

df_round1.to_csv(
    "googlescholar_round1_all.csv",
    index=False
)

# Save successful results only
df_scholar_success = df_round1[
    df_round1["google_scholar_status"] == "Success"
].copy()

df_scholar_success.to_csv(
    "googlescholar_all_success.csv",
    index=False
)

print(f"\nTotal attempted: {len(df_round1)}")
print(f"Successful: {len(df_scholar_success)}")
print(f"Failed: {len(df_round1) - len(df_scholar_success)}")

Processing paper 1/205...
Consecutive failures: 1


KeyboardInterrupt: 

#### Identify the remaining papers

In [19]:
# Load cumulative successful results
df_scholar_success = pd.read_csv("googlescholar_all_success.csv")

# Identify papers not successfully scraped
successful_dois = set(df_scholar_success["doi"].dropna())

df_remaining = df_filtered[
    ~df_filtered["doi"].isin(successful_dois)
].copy().reset_index(drop=True)

print(f"Total papers: {len(df_filtered)}")
print(f"Successfully retrieved: {len(df_scholar_success)}")
print(f"Remaining: {len(df_remaining)}")

Total papers: 205
Successfully retrieved: 211
Remaining: 10


#### Rerun the next round

In [20]:
driver = webdriver.Chrome()

new_scholar_results = []

consecutive_failures = 0
max_consecutive_failures = 3

for i, row in df_remaining.iterrows():

    print(f"Processing paper {i + 1}/{len(df_remaining)}...")

    result = get_scholar_data(driver, row["doi"])
    result["doi"] = row["doi"]

    new_scholar_results.append(result)

    # Check status
    if result["google_scholar_status"] == "Success":
        consecutive_failures = 0
    else:
        consecutive_failures += 1
        print(f"Consecutive failures: {consecutive_failures}")

    # Stop after 3 consecutive failures
    if consecutive_failures > max_consecutive_failures:
        print("More than 3 consecutive failures. Stopping scraping.")
        break

    time.sleep(random.uniform(5, 15))

driver.quit()

Processing paper 1/10...
Consecutive failures: 1
Processing paper 2/10...
Consecutive failures: 2
Processing paper 3/10...
Consecutive failures: 3
Processing paper 4/10...
Processing paper 5/10...
Processing paper 6/10...
Processing paper 7/10...
Processing paper 8/10...
Processing paper 9/10...
Processing paper 10/10...


In [21]:
# Convert Round 2 results into a DataFrame
df_round2 = pd.DataFrame(new_scholar_results)

# Save all Round 2 attempts
df_round2.to_csv(
    "googlescholar_round2_all.csv",
    index=False
)

# Extract successful results
df_round2_success = df_round2[
    df_round2["google_scholar_status"] == "Success"
].copy()

# Combine with previous successful results
df_scholar_success = pd.concat(
    [df_scholar_success, df_round2_success],
    ignore_index=True
)

# Remove duplicate papers
df_scholar_success = df_scholar_success.drop_duplicates(
    subset="doi",
    keep="first"
)

# Update master CSV
df_scholar_success.to_csv(
    "googlescholar_all_success.csv",
    index=False
)

print(f"Total successful papers: {len(df_scholar_success)}")

Total successful papers: 218


In [22]:
successful_dois = set(df_scholar_success["doi"].dropna())

df_remaining = df_filtered[
    ~df_filtered["doi"].isin(successful_dois)
].copy().reset_index(drop=True)

print(f"Papers remaining: {len(df_remaining)}")

Papers remaining: 3


### 4. Merge the whole dataset

In [24]:
df_final = df_filtered.merge(
    df_scholar_success,
    on="doi",
    how="inner"
)

df_final = df_final.drop(
    columns=['matched_keywords', 'abstract_openalex', 'google_scholar_title', 'google_scholar_status'])

df_final

,title,journal_code,year,authors,doi,google_scholar_abstract,google_scholar_citations
0,Made With AI: Consumer Engagement With Social ...,JCR,2026,"[Stephan Carney, Ignacio Riveros, Stephanie M....",https://doi.org/10.1093/jcr/ucag013,"Social media shapes how people connect, commun...",19.0
1,AI Companions Reduce Loneliness,JCR,2025,"[Julian De Freitas, Zeliha Uğuralp, Ahmet Kaan...",https://doi.org/10.1093/jcr/ucaf040,Chatbots are now able to engage in sophisticat...,560.0
2,Collaborating with Generative AI in Consumer C...,JCR,2025,"[Amber M. Epp, Ashlee Humphreys]",https://doi.org/10.1093/jcr/ucaf014,Generative Artificial Intelligence (GenAI) has...,28.0
3,GenAI and Consumer Research: Are We the Last G...,JCR,2025,[Bernd H. Schmitt],https://doi.org/10.1093/jcr/ucaf015,Explorations of Generative artificial intellig...,8.0
4,Ideation with Generative AI—in Consumer Resear...,JCR,2025,"[Julian De Freitas, Gideon Nave, Stefano Puntoni]",https://doi.org/10.1093/jcr/ucaf012,The use of generative AI (genAI) in consumer r...,89.0
...,...,...,...,...,...,...,...
197,Writing More Compelling Creative Appeals: A De...,MS,2022,"[Jiyeon Hong, Paul R. Hoban]",https://doi.org/10.1287/mksc.2022.1351,We present a deep learning algorithm to provid...,42.0
198,Large Language Models and Creative Content Des...,QME,2026,"[Jean-Pierre Dubé, Ariel Xu]",https://doi.org/10.1007/s11129-025-09303-9,A sequence of three randomized controlled tria...,7.0
199,Privacy and prediction: how useful are geo-tra...,QME,2025,"[Unnati Narang, Fernando Luco]",https://doi.org/10.1007/s11129-025-09299-2,Accurately predicting consumer visits is criti...,3.0
200,Complementarities between algorithmic and huma...,QME,2024,"[Michael Allan Ribers, Hannes Ullrich]",https://doi.org/10.1007/s11129-024-09284-1,Artificial Intelligence has the potential to i...,12.0


In [55]:
df_final.to_csv('test.csv')